In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [11]:
pd.set_option('display.max_rows', None)

filepath = r"C:\Users\felix\ames_housing-ml\data\AmesHousing.csv"
df =pd.read_csv(filepath)
df.drop(columns=['Order', 'PID', 'Mo Sold', 'Yr Sold'], inplace=True)
df.head()
df.tail()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Sale Type,Sale Condition,SalePrice
2925,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,AllPub,CulDSac,...,0,0,0,NaN,GdPrv,NaN,0,WD,Normal,142500
2926,20,RL,NaN,8885,Pave,NaN,IR1,Low,AllPub,Inside,...,0,0,0,NaN,MnPrv,NaN,0,WD,Normal,131000
2927,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,0,NaN,MnPrv,Shed,700,WD,Normal,132000
2928,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,0,NaN,NaN,NaN,0,WD,Normal,170000
2929,60,RL,74.0,9627,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,0,NaN,NaN,NaN,0,WD,Normal,188000


In [12]:
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
categorical_cols = X.select_dtypes(include='object').columns

high_cardinality = [
    col for col in categorical_cols
    if X[col].nunique() > 20
]

low_cardinality = [
    col for col in categorical_cols
    if X[col].nunique() <= 20
]

numeric_cols = X.select_dtypes(exclude='object').columns

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        
        ('num', SimpleImputer(strategy='median'), numeric_cols),

        
        ('high_card', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ))
        ]), high_cardinality),

        
        ('low_card', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ))
        ]), low_cardinality)
    ]
)

In [15]:
model = RandomForestRegressor(
    n_estimators=700,
    max_depth=8,
    random_state=42
)

In [16]:
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', model)
])

In [17]:
pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_valid)

mean_absolute_error = mean_absolute_error(y_valid, preds)
print(f"Mean Absolute Error: {mean_absolute_error}")

Mean Absolute Error: 16906.4222476674
